In [1]:
%load_ext autoreload
%autoreload 2
%matplotlib inline
import os
import sys
os.environ["CUDA_VISIBLE_DEVICES"] = "0"
sys.path.append("/home/grega/geoloc")

In [25]:
import torch
import rasterio
import numpy as np

import geopandas as gpd
import matplotlib.pyplot as plt
import torchvision.transforms.functional as TF

from shapely.affinity import rotate
from rasterio.windows import from_bounds

from rasterio.features import geometry_mask
from geoloc.data.utils import load_image
from shapely.geometry import box
from numba import jit

In [ ]:
@jit(nopython=True)
def largest_rectangle_in_mask(mask):
	H, W = mask.shape
	height = np.zeros((H, W), dtype=np.int32)
	
	# Calculate heights
	for i in range(H):
		for j in range(W):
			if mask[i, j]:
				if i > 0:
					height[i, j] = height[i-1, j] + 1
				else:
					height[i, j] = 1
			else:
				height[i, j] = 0
	
	max_area = 0
	max_rect = (0, 0, 0, 0)
	
	for i in range(H):
		area, rect = _histogram_numba(height[i], i)
		if area > max_area:
			max_area = area
			max_rect = rect
	
	return max_rect

@jit(nopython=True)
def _histogram_numba(heights, row_idx):
	n = len(heights)
	stack = np.empty(n, dtype=np.int32)
	stack_size = 0
	max_area = 0
	best_top = 0
	best_left = 0
	best_height = 0
	best_width = 0
	
	for j in range(n):
		while stack_size > 0 and heights[j] < heights[stack[stack_size-1]]:
			stack_size -= 1
			h_idx = stack[stack_size]
			h = heights[h_idx]
			if stack_size == 0:
				w = j
				left = 0
			else:
				w = j - stack[stack_size-1] - 1
				left = stack[stack_size-1] + 1
			
			area = h * w
			if area > max_area:
				max_area = area
				best_top = row_idx - h + 1
				best_left = left
				best_height = h
				best_width = w
		
		stack[stack_size] = j
		stack_size += 1
	
	while stack_size > 0:
		stack_size -= 1
		h_idx = stack[stack_size]
		h = heights[h_idx]
		if stack_size == 0:
			w = n
			left = 0
		else:
			w = n - stack[stack_size-1] - 1
			left = stack[stack_size-1] + 1
		
		area = h * w
		if area > max_area:
			max_area = area
			best_top = row_idx - h + 1
			best_left = left
			best_height = h
			best_width = w
	
	return max_area, (best_top, best_left, best_height, best_width)


# No rotation alignment between drone and reference images

In [ ]:
visloc_datadir = "/storage/datasets/AerialLoc/UAV_VisLoc/"
plot = False
save = True

for subdir in ["01", "02", "03", "04", "05", "06", "08", "09", "10", "11"]:
	visloc_subdir = os.path.join(visloc_datadir, subdir)
	drone_boxes = gpd.read_file(os.path.join(visloc_subdir, "drone_boxes.geojson"))
	satellite = rasterio.open(os.path.join(visloc_subdir, f"satellite{subdir}.tif"))
	_csv = gpd.read_file(os.path.join(visloc_subdir, f"{subdir}.csv"))

	for i, drone_box in enumerate(drone_boxes.itertuples()):
		roll, pitch, yaw = _csv.iloc[i][["Omega", "Kappa", "Phi1"]].astype(float)
		drone_image_path = os.path.join(visloc_subdir, "drone", drone_box.filename)
		drone_image = load_image(drone_image_path)

		geom = drone_box.geometry
		minx, miny, maxx, maxy = geom.bounds
		window = from_bounds(minx, miny, maxx, maxy, transform=satellite.transform)

		subset = satellite.read(window=window)
		subset = torch.from_numpy(subset)
		extent = (minx, maxx, miny, maxy)

		geom_mapping = [geom.__geo_interface__]
		mask = geometry_mask(
			geom_mapping,
			invert=True,                   # invert=True → True inside polygon
			out_shape=subset.shape[1:],    # H, W
			transform=satellite.window_transform(window) if 'window' in locals() else satellite.transform,
			all_touched=True
		)
		mask = torch.from_numpy(mask)  # H, W

		if plot:
			fig, axs = plt.subplots(1, 3, figsize=(24, 12))
			axs[0].imshow(drone_image.permute(1, 2, 0) / 255.0)
			axs[0].set_title("Drone image")
			axs[1].imshow(subset.permute(1,2,0))
			axs[1].set_title("Satellite subset with drone footprint")
			axs[2].imshow(mask.squeeze(0), cmap='gray')
			axs[2].set_title("Mask")
			plt.show()

		if save:
			visloc_train_savedir = os.path.join(visloc_datadir, subdir, "drone_ref_pairs")
			# visloc_train_drone_savedir = os.path.join(visloc_train_savedir, "not_aligned", "drone")
			visloc_train_reference_savedir = os.path.join(visloc_train_savedir, "not_aligned", "reference")
			visloc_train_drone_masks_savedir = os.path.join(visloc_train_savedir, "not_aligned", "drone_masks")
			# os.makedirs(visloc_train_drone_savedir, exist_ok=True)
			os.makedirs(visloc_train_reference_savedir, exist_ok=True)
			os.makedirs(visloc_train_drone_masks_savedir, exist_ok=True)

			# drone_savepath = os.path.join(visloc_train_drone_savedir, f"{drone_box.filename}")
			reference_savepath = os.path.join(visloc_train_reference_savedir, f"{drone_box.filename}")
			mask_savepath = os.path.join(visloc_train_drone_masks_savedir, f"{os.path.splitext(drone_box.filename)[0]}.npy")

			# TF.to_pil_image(drone_image.to(torch.uint8)).save(drone_savepath)
			TF.to_pil_image(subset).save(reference_savepath)
			np.save(mask_savepath, mask.numpy())

		if i >= 0:
			break
	break

# North align drone images to match reference images

In [33]:
visloc_datadir = "/storage/datasets/AerialLoc/UAV_VisLoc/"
plot = False
save = True

for subdir in ["01", "02", "03", "04", "05", "06", "08", "09", "10", "11"]:
# for subdir in ["10", "11"]:
	visloc_subdir = os.path.join(visloc_datadir, subdir)
	drone_boxes = gpd.read_file(os.path.join(visloc_subdir, "drone_boxes.geojson"))
	satellite = rasterio.open(os.path.join(visloc_subdir, f"satellite{subdir}.tif"))
	_csv = gpd.read_file(os.path.join(visloc_subdir, f"{subdir}.csv"))

	for i, drone_box in enumerate(drone_boxes.itertuples()):
		# if i <= 9:
		# 	continue
		roll, pitch, yaw = _csv.iloc[i][["Omega", "Kappa", "Phi1"]].astype(float)
		drone_image_path = os.path.join(visloc_subdir, "drone", drone_box.filename)
		drone_image = load_image(drone_image_path)

		north_aligned_drone_image = TF.rotate(drone_image, angle=-yaw, expand=True, fill=0)
		drone_image = north_aligned_drone_image

		geom = drone_box.geometry
		minx, miny, maxx, maxy = geom.bounds
		window = from_bounds(minx, miny, maxx, maxy, transform=satellite.transform)

		subset = satellite.read(window=window)
		subset = torch.from_numpy(subset)
		# mask = torch.where(drone_image.sum(dim=0) > 0, True, False)
		mask = drone_image.sum(dim=0) > 0
		# fig, axes = plt.subplots(1,2, figsize=(12,6))
		# axes[0].imshow(drone_image.permute(1,2,0) / 255.0)
		# axes[1].imshow(mask.numpy(), cmap='gray')
		# plt.show()
		# # geom_mapping = [geom.__geo_interface__]
		# # mask = geometry_mask(
		# # 	geom_mapping,
		# # 	invert=True,                   # invert=True → True inside polygon
		# # 	out_shape=subset.shape[1:],    # H, W
		# # 	transform=satellite.window_transform(window) if 'window' in locals() else satellite.transform,
		# # 	all_touched=True
		# # )
		# # mask = torch.from_numpy(mask)  # H, W
		resize_size = (subset.shape[1], subset.shape[2])
		subset = TF.resize(subset, size=resize_size)
		drone_image = TF.resize(drone_image, size=resize_size)
		mask = TF.resize(mask.unsqueeze(0).float(), size=resize_size).squeeze(0).bool()

		subset_masked = subset.clone()  # or subset.copy() if NumPy
		subset_masked[:, ~mask] = 0     # zero out pixels outside polygon
		subset = subset_masked
		extent = (minx, maxx, miny, maxy)

		mask_np = mask.numpy()
		top, left, h_crop, w_crop = largest_rectangle_in_mask(mask_np)

		subset = subset[:, top:top+h_crop, left:left+w_crop]
		# fig, axes = plt.subplots(1,2, figsize=(12,6))
		# axes[0].imshow(drone_image.permute(1,2,0) / 255.0)
		# axes[1].imshow(mask.numpy(), cmap='gray')
		# plt.show()
		drone_image = drone_image[:, top:top+h_crop, left:left+w_crop]
		mask = mask[top:top+h_crop, left:left+w_crop]
		# fig, axes = plt.subplots(1,2, figsize=(12,6))
		# axes[0].imshow(drone_image.permute(1,2,0) / 255.0)
		# axes[1].imshow(mask.numpy(), cmap='gray')
		# plt.show()
		if plot:
			fig, axs = plt.subplots(1, 2, figsize=(24, 12))
			axs[0].imshow(drone_image.permute(1, 2, 0) / 255.0)
			axs[0].set_title("Drone image")
			axs[1].imshow(subset.permute(1,2,0))
			axs[1].set_title("Satellite subset with drone footprint")
			plt.show()

		if save:
			visloc_train_savedir = os.path.join(visloc_datadir, subdir, "drone_ref_pairs")
			visloc_train_drone_savedir = os.path.join(visloc_train_savedir, "north_aligned", "drone")
			visloc_train_reference_savedir = os.path.join(visloc_train_savedir, "north_aligned", "reference")
			os.makedirs(visloc_train_drone_savedir, exist_ok=True)
			os.makedirs(visloc_train_reference_savedir, exist_ok=True)

			drone_savepath = os.path.join(visloc_train_drone_savedir, f"{drone_box.filename}")
			reference_savepath = os.path.join(visloc_train_reference_savedir, f"{drone_box.filename}")

			TF.to_pil_image(drone_image.to(torch.uint8)).save(drone_savepath)
			TF.to_pil_image(subset).save(reference_savepath)

	# 	if i >= 13:
	# 		break
	# break


# Rotate reference images to align with drone images (DO NOT USE; USELESS)

In [11]:
visloc_datadir = "/storage/datasets/AerialLoc/UAV_VisLoc/"
plot = False
save = True

for subdir in ["01", "02", "03", "04", "05", "06", "08", "09", "10", "11"]:
	visloc_subdir = os.path.join(visloc_datadir, subdir)
	drone_boxes = gpd.read_file(os.path.join(visloc_subdir, "drone_boxes.geojson"))
	satellite = rasterio.open(os.path.join(visloc_subdir, f"satellite{subdir}.tif"))
	_csv = gpd.read_file(os.path.join(visloc_subdir, f"{subdir}.csv"))

	for i, drone_box in enumerate(drone_boxes.itertuples()):
		roll, pitch, yaw = _csv.iloc[i][["Omega", "Kappa", "Phi1"]].astype(float)
		drone_image_path = os.path.join(visloc_subdir, "drone", drone_box.filename)
		drone_image = load_image(drone_image_path)

		geom = drone_box.geometry
		minx, miny, maxx, maxy = geom.bounds
		eps = 0.0001
		window = from_bounds(minx-eps, miny-eps, maxx+eps, maxy+eps, transform=satellite.transform)
		extent = (minx-eps, maxx+eps, miny-eps, maxy+eps)

		subset = satellite.read(window=window)
		transform = satellite.window_transform(window)
		crs = satellite.crs

		subset = torch.from_numpy(subset)
		subset = TF.rotate(
			subset, 
			angle=yaw, 
			expand=False, 
			fill=0
		)

		geom = rotate(drone_box.geometry, yaw, origin='center', use_radians=False)
		geom = box(*geom.bounds)
		# plt.imshow(subset.permute(1,2,0), extent=extent)
		# plt.plot(geom.exterior.xy[0], geom.exterior.xy[1], color='red')

		minx, miny, maxx, maxy = geom.bounds
		window_inner = from_bounds(minx, miny, maxx, maxy, transform=transform)

		r0 = int(window_inner.row_off)
		r1 = int(window_inner.row_off + window_inner.height)
		c0 = int(window_inner.col_off)
		c1 = int(window_inner.col_off + window_inner.width)

		subset_cropped = subset[:, r0:r1, c0:c1]
		if plot:
			fig, axs = plt.subplots(1, 2, figsize=(16, 8))
			axs[0].imshow(subset_cropped.permute(1,2,0), extent=(minx, maxx, miny, maxy))
			axs[0].set_title("Cropped Rotated Satellite Subset")
			axs[1].imshow(drone_image.permute(1,2,0) / 255.0, extent=(minx, maxx, miny, maxy))
			axs[1].set_title("Drone Image Overlay")
			plt.show()
		
		if save:
			visloc_train_savedir = os.path.join(visloc_datadir, subdir, "drone_ref_pairs")
			# visloc_train_drone_savedir = os.path.join(visloc_train_savedir, "drone_aligned", "drone")
			visloc_train_reference_savedir = os.path.join(visloc_train_savedir, "drone_aligned", "reference")
			# os.makedirs(visloc_train_drone_savedir, exist_ok=True)
			os.makedirs(visloc_train_reference_savedir, exist_ok=True)

			# drone_savepath = os.path.join(visloc_train_drone_savedir, f"{drone_box.filename}")
			reference_savepath = os.path.join(visloc_train_reference_savedir, f"{drone_box.filename}")

			# TF.to_pil_image(drone_image.to(torch.uint8)).save(drone_savepath)
			TF.to_pil_image(subset_cropped).save(reference_savepath)

	# 	if i >= 0:
	# 		break
	# break

KeyboardInterrupt: 